# AURION — Robustez y análisis de errores

Dos análisis sobre el modelo `yolo11n` ya entrenado con el split agrupado.

**1. Robustez fotométrica.** AURION asume una cámara fija en un puesto de control:
encuadre y ángulo constantes. Ese supuesto es razonable, pero deja abiertas las
variables fotométricas de una cámara industrial real. Aquí se degrada el test set
en niveles controlados y se mide la caída de mAP.

**2. Análisis de errores.** Empareja predicciones con verdad terreno y clasifica
cada fallo. Prioriza los `palet_roto` no detectados, que en control de calidad
son el error caro: dejan pasar producto defectuoso.

> Todas las degradaciones son fotométricas o desenfoque leve. **No mueven los
> objetos**, así que las cajas de las etiquetas siguen siendo válidas y no hay
> que tocar ningún `.txt`. Rotaciones o recortes las invalidarían.

---
**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → GPU.

## 1. Preparación

In [ ]:
!pip install -q ultralytics
import ultralytics; print("ultralytics", ultralytics.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Si ya tienes todo en /content de esta sesión, sáltate esta celda.

### Configuración

Ajusta estas rutas. Es lo único que hay que tocar en todo el notebook.

In [ ]:
from pathlib import Path

PESOS = Path("/content/runs/yolo11n/weights/best.pt")
SPLIT = Path("/content/aurion_split")      # contiene test/images y test/labels
OUT   = Path("/content/analisis")

CLASES = [
    "palet_bueno",
    "palet_roto",
    "paquete_emb_correct_dim_correct",
    "paquete_emb_correct_dim_incorrect",
    "paquete_emb_incorrect_dim_correct",
    "paquete_emb_incorrect_dim_incorrect",
]

CRITICAS = {"palet_roto"}   # falso negativo aquí = defecto que pasa el control
IOU_MIN  = 0.5
CONF_MIN = 0.25
IMGSZ    = 640
EXT_IMG  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

OUT.mkdir(parents=True, exist_ok=True)

# Comprobación
assert PESOS.exists(),                 f"No existe {PESOS}"
assert (SPLIT/"test"/"images").is_dir(), f"No existe {SPLIT}/test/images"
n = len(list((SPLIT/"test"/"images").iterdir()))
print(f"OK — pesos encontrados y {n} imágenes en test")

In [ ]:
import cv2, shutil, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO

np.random.seed(0)
model = YOLO(str(PESOS))
print("Modelo cargado")

---
## 2. Línea base

El número de referencia contra el que se compara todo lo demás.

In [ ]:
r0 = model.val(data=str(SPLIT/"data.yaml"), split="test",
               imgsz=IMGSZ, verbose=False, plots=False)
BASE50, BASE5095 = float(r0.box.map50), float(r0.box.map)

print(f"mAP50    = {BASE50:.4f}")
print(f"mAP50-95 = {BASE5095:.4f}\n")
for i, c in enumerate(CLASES):
    print(f"  {c:<40} {float(r0.box.maps[i]):.4f}")

---
## 3. Robustez fotométrica

### Definición de las degradaciones

La más relevante para ti es **movimiento**: simula el palet avanzando por la cinta
mientras la cámara dispara. Es la degradación más realista en un control en línea
y ninguna imagen generada la tiene.

In [ ]:
def brillo(img, f):
    """Multiplica la luminancia. f<1 oscurece, f>1 aclara."""
    return np.clip(img.astype(np.float32) * f, 0, 255).astype(np.uint8)

def contraste(img, f):
    """Comprime el rango dinámico alrededor de la media."""
    m = img.mean()
    return np.clip((img.astype(np.float32) - m) * f + m, 0, 255).astype(np.uint8)

def ruido(img, sigma):
    """Ruido gaussiano: sensor con poca luz o ganancia alta."""
    if sigma == 0: return img
    return np.clip(img.astype(np.float32) + np.random.normal(0, sigma, img.shape),
                   0, 255).astype(np.uint8)

def desenfoque(img, k):
    """Desenfoque gaussiano: mal enfoque."""
    if k <= 1: return img
    k = int(k) | 1
    return cv2.GaussianBlur(img, (k, k), 0)

def movimiento(img, k):
    """Desenfoque de movimiento horizontal: palet avanzando en la cinta."""
    if k <= 1: return img
    k = int(k)
    kern = np.zeros((k, k), np.float32); kern[k//2, :] = 1.0/k
    return cv2.filter2D(img, -1, kern)

def compresion(img, q):
    """Recompresión JPEG: cámara IP con ancho de banda limitado."""
    ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), int(q)])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR) if ok else img

DEGRADACIONES = {
    "brillo":     (brillo,     [1.0, 0.8, 0.6, 0.4, 1.2, 1.5], "factor de brillo"),
    "contraste":  (contraste,  [1.0, 0.8, 0.6, 0.4],           "factor de contraste"),
    "ruido":      (ruido,      [0, 5, 10, 20, 35],             "sigma"),
    "desenfoque": (desenfoque, [1, 3, 5, 9, 15],               "kernel"),
    "movimiento": (movimiento, [1, 5, 11, 21, 31],             "kernel (px)"),
    "compresion": (compresion, [100, 70, 50, 30, 15],          "calidad JPEG"),
}
print("6 degradaciones definidas")

### Vista previa

Comprueba a ojo que los niveles son razonables antes de gastar diez minutos de GPU.

In [ ]:
ejemplo = sorted((SPLIT/"test"/"images").iterdir())[0]
img0 = cv2.imread(str(ejemplo))

fig, axes = plt.subplots(len(DEGRADACIONES), 5, figsize=(16, 3*len(DEGRADACIONES)))
for fila, (nom, (fn, niveles, eje)) in zip(axes, DEGRADACIONES.items()):
    for ax, niv in zip(fila, niveles[:5]):
        ax.imshow(cv2.cvtColor(fn(img0.copy(), niv), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{nom}={niv}", fontsize=9); ax.axis("off")
    for ax in fila[len(niveles):]:
        ax.axis("off")
plt.tight_layout(); plt.show()

### Ejecución

Unos 10 minutos en A100.

In [ ]:
def construir(fn, nivel, dst):
    (dst/"test"/"images").mkdir(parents=True, exist_ok=True)
    (dst/"test"/"labels").mkdir(parents=True, exist_ok=True)
    for f in sorted((SPLIT/"test"/"images").iterdir()):
        if f.suffix.lower() not in EXT_IMG: continue
        im = cv2.imread(str(f))
        if im is None: continue
        cv2.imwrite(str(dst/"test"/"images"/f.name), fn(im, nivel))
        txt = SPLIT/"test"/"labels"/(f.stem + ".txt")
        if txt.exists(): shutil.copy2(txt, dst/"test"/"labels"/txt.name)
    (dst/"data.yaml").write_text(
        f"path: {dst.as_posix()}\ntrain: test/images\nval: test/images\n"
        f"test: test/images\n\nnc: {len(CLASES)}\nnames: {CLASES}\n")
    return dst/"data.yaml"

filas = []
for nom, (fn, niveles, eje) in DEGRADACIONES.items():
    print(f"\n{'='*50}\n{nom.upper()}\n{'='*50}")
    for niv in niveles:
        dst = Path("/content/_tmp_deg")
        if dst.exists(): shutil.rmtree(dst)
        yml = construir(fn, niv, dst)
        r = model.val(data=str(yml), split="test", imgsz=IMGSZ,
                      verbose=False, plots=False)
        fila = {"degradacion": nom, "eje": eje, "nivel": niv,
                "mAP50": round(float(r.box.map50), 4),
                "mAP50_95": round(float(r.box.map), 4),
                "precision": round(float(r.box.mp), 4),
                "recall": round(float(r.box.mr), 4)}
        for i, c in enumerate(CLASES):
            fila[f"mAP_{c}"] = round(float(r.box.maps[i]), 4)
        filas.append(fila)
        print(f"  {eje}={niv:<6} mAP50 {fila['mAP50']:.4f}  "
              f"mAP50-95 {fila['mAP50_95']:.4f}  R {fila['recall']:.3f}")
        shutil.rmtree(dst)

rob = pd.DataFrame(filas)
rob.to_csv(OUT/"robustez.csv", index=False)
print(f"\nGuardado en {OUT/'robustez.csv'}")

### Curvas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (nom, (_, _, eje)) in zip(axes.ravel(), DEGRADACIONES.items()):
    s = rob[rob.degradacion == nom]
    ax.plot(range(len(s)), s.mAP50, "o-", lw=2, label="mAP50")
    ax.plot(range(len(s)), s.mAP50_95, "s--", lw=2, label="mAP50-95")
    ax.axhline(BASE50, color="gray", ls=":", lw=1, label="base")
    ax.set_xticks(range(len(s))); ax.set_xticklabels([str(v) for v in s.nivel])
    ax.set_title(nom, fontsize=12); ax.set_xlabel(eje, fontsize=9)
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.suptitle("AURION — robustez del yolo11n frente a degradaciones fotométricas",
             fontsize=14)
plt.tight_layout(); plt.savefig(OUT/"robustez.png", dpi=150); plt.show()

### Rango operativo

Aquí está el resultado que va a la web: **hasta qué nivel de cada degradación
aguanta el sistema** antes de perder más del 10% de mAP. Eso es una especificación
de despliegue, no una métrica suelta.

In [ ]:
print(f"Base: mAP50 = {BASE50:.4f}\n")
print(f"{'degradación':<13} {'peor nivel':>11} {'mAP50':>8} {'caída':>9}   límite aceptable")
print("-"*72)
for nom, (_, _, eje) in DEGRADACIONES.items():
    s = rob[rob.degradacion == nom]
    peor = s.loc[s.mAP50.idxmin()]
    ok = s[s.mAP50 >= BASE50*0.9]
    lim = f"{eje} hasta {ok.nivel.iloc[-1]}" if len(ok) else "ninguno"
    print(f"{nom:<13} {str(peor.nivel):>11} {peor.mAP50:>8.4f} "
          f"{100*(peor.mAP50-BASE50)/BASE50:>8.1f}%   {lim}")

---
## 4. Análisis de errores

Cuatro categorías:

| | |
|---|---|
| **ACIERTO** | predicho, clase correcta |
| **CONFUSIÓN** | sitio correcto, clase equivocada |
| **NO_DETECTADO** | objeto real que el modelo no vio (falso negativo) |
| **FANTASMA** | detección sin objeto real detrás (falso positivo) |

In [ ]:
def leer_gt(txt, w, h):
    cajas = []
    if not txt.exists(): return cajas
    for l in txt.read_text().splitlines():
        p = l.split()
        if len(p) < 5: continue
        c = int(p[0]); cx, cy, bw, bh = map(float, p[1:5])
        cajas.append((c, (cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h))
    return cajas

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x2-x1) * max(0., y2-y1)
    if inter <= 0: return 0.
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.

def emparejar(gts, preds):
    cand = sorted(((iou(g[1:], p[1:5]), i, j)
                   for i, g in enumerate(gts) for j, p in enumerate(preds)
                   if iou(g[1:], p[1:5]) >= IOU_MIN), reverse=True)
    par, ug, up = [], set(), set()
    for v, i, j in cand:
        if i in ug or j in up: continue
        par.append((i, j, v)); ug.add(i); up.add(j)
    return (par, [i for i in range(len(gts)) if i not in ug],
                 [j for j in range(len(preds)) if j not in up])

def dibujar(img, gts, preds, rg=(), rp=()):
    o = img.copy()
    for i, (c, x1, y1, x2, y2) in enumerate(gts):
        cv2.rectangle(o, (int(x1), int(y1)), (int(x2), int(y2)),
                      (0,200,0), 4 if i in rg else 1)
        if i in rg:
            cv2.putText(o, f"REAL: {CLASES[c]}", (int(x1), max(14, int(y1)-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, .45, (0,200,0), 1, cv2.LINE_AA)
    for j, (c, x1, y1, x2, y2, cf) in enumerate(preds):
        cv2.rectangle(o, (int(x1), int(y1)), (int(x2), int(y2)),
                      (0,0,235), 4 if j in rp else 1)
        if j in rp:
            cv2.putText(o, f"PRED: {CLASES[c]} {cf:.2f}",
                        (int(x1), min(o.shape[0]-6, int(y2)+16)),
                        cv2.FONT_HERSHEY_SIMPLEX, .45, (0,0,235), 1, cv2.LINE_AA)
    return o

print("Funciones listas")

In [ ]:
for c in ["no_detectado", "confusion", "fantasma"]:
    (OUT/c).mkdir(parents=True, exist_ok=True)

errores, guardadas = [], {}
imgs = [f for f in sorted((SPLIT/"test"/"images").iterdir())
        if f.suffix.lower() in EXT_IMG]

for f in imgs:
    img = cv2.imread(str(f))
    if img is None: continue
    h, w = img.shape[:2]
    gts = leer_gt(SPLIT/"test"/"labels"/(f.stem+".txt"), w, h)
    r = model.predict(str(f), imgsz=IMGSZ, conf=CONF_MIN, verbose=False)[0]
    preds = [(int(b.cls[0]), *b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]

    par, gs, ps = emparejar(gts, preds)
    aqui = []
    for i, j, v in par:
        if gts[i][0] != preds[j][0]:
            aqui.append({"imagen": f.name, "tipo": "CONFUSION",
                         "clase_real": CLASES[gts[i][0]], "clase_pred": CLASES[preds[j][0]],
                         "conf": round(preds[j][5],3), "iou": round(v,3),
                         "critico": CLASES[gts[i][0]] in CRITICAS, "_g": i, "_p": j})
    for i in gs:
        aqui.append({"imagen": f.name, "tipo": "NO_DETECTADO",
                     "clase_real": CLASES[gts[i][0]], "clase_pred": "-",
                     "conf": None, "iou": 0.,
                     "critico": CLASES[gts[i][0]] in CRITICAS, "_g": i, "_p": None})
    for j in ps:
        aqui.append({"imagen": f.name, "tipo": "FANTASMA", "clase_real": "-",
                     "clase_pred": CLASES[preds[j][0]], "conf": round(preds[j][5],3),
                     "iou": 0., "critico": False, "_g": None, "_p": j})

    if aqui:
        errores += aqui
        peor = max(aqui, key=lambda x: (x["critico"], x["tipo"]=="NO_DETECTADO"))
        carp = {"NO_DETECTADO":"no_detectado","CONFUSION":"confusion",
                "FANTASMA":"fantasma"}[peor["tipo"]]
        rg = {x["_g"] for x in aqui if x["_g"] is not None}
        rp = {x["_p"] for x in aqui if x["_p"] is not None}
        pref = "CRITICO_" if any(x["critico"] for x in aqui) else ""
        ruta = OUT/carp/f"{pref}{f.stem}.jpg"
        cv2.imwrite(str(ruta), dibujar(img, gts, preds, rg, rp))
        guardadas[f.name] = (ruta, peor["critico"])

err = pd.DataFrame(errores).drop(columns=["_g","_p"])
err.to_csv(OUT/"errores.csv", index=False)
print(f"{len(err)} errores en {err.imagen.nunique()} imágenes de {len(imgs)}")

### Resumen

In [ ]:
print("POR TIPO"); print(err.tipo.value_counts().to_string())

crit = err[err.critico & (err.tipo=="NO_DETECTADO")]
print(f"\nCRÍTICOS — palet_roto no detectados: {len(crit)}")
if len(crit): print(f"  en {crit.imagen.nunique()} imágenes distintas")

print("\nPOR CLASE REAL")
print(err[err.clase_real!="-"].pivot_table(index="clase_real", columns="tipo",
      values="imagen", aggfunc="count", fill_value=0).to_string())

print("\nCONFUSIONES MÁS FRECUENTES")
for (a,b), n in err[err.tipo=="CONFUSION"].groupby(
        ["clase_real","clase_pred"]).size().sort_values(ascending=False).head(8).items():
    print(f"  {n:>3}x  {a}  ->  {b}")

fant = err[err.tipo=="FANTASMA"]
if len(fant):
    print("\nFANTASMAS POR CLASE")
    print(fant.clase_pred.value_counts().to_string())
    print(f"\n  Confianza media: {fant.conf.mean():.3f}")
    print("  Si es baja, subir CONF_MIN los elimina sin perder aciertos.")

### Los fallos críticos, en imagen

Verde = objeto real. Rojo = predicción del modelo.

**Agrupa estos a mano por causa**: oclusión, palet pequeño o lejano, defecto poco
visible, confusión con la textura del fondo. Con cuatro o cinco categorías y dos
ejemplos de cada una tienes la sección de análisis de errores lista.

In [ ]:
criticas = [(n,r) for n,(r,c) in guardadas.items() if c][:12]
if criticas:
    filas = (len(criticas)+2)//3
    fig, axes = plt.subplots(filas, 3, figsize=(16, 4.5*filas))
    for ax, (nom, ruta) in zip(np.atleast_1d(axes).ravel(), criticas):
        ax.imshow(cv2.cvtColor(cv2.imread(str(ruta)), cv2.COLOR_BGR2RGB))
        ax.set_title(nom[:42], fontsize=8); ax.axis("off")
    for ax in np.atleast_1d(axes).ravel()[len(criticas):]: ax.axis("off")
    plt.suptitle("palet_roto NO DETECTADOS — el error caro", fontsize=14)
    plt.tight_layout(); plt.savefig(OUT/"criticos.png", dpi=130); plt.show()
else:
    print("Sin fallos críticos de palet_roto con este umbral.")

### Barrido de confianza

¿Cuánto ganas subiendo el umbral? Menos fantasmas, pero también menos detecciones
reales. Este barrido te dice dónde está el punto de equilibrio, y esa es una
decisión de diseño que puedes justificar en la web.

In [ ]:
sweep = []
for c in [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]:
    r = model.val(data=str(SPLIT/"data.yaml"), split="test", imgsz=IMGSZ,
                  conf=c, verbose=False, plots=False)
    sweep.append({"conf": c, "mAP50": round(float(r.box.map50),4),
                  "precision": round(float(r.box.mp),4),
                  "recall": round(float(r.box.mr),4),
                  "recall_palet_roto": round(float(r.box.r[1]),4)})
sw = pd.DataFrame(sweep); sw.to_csv(OUT/"barrido_conf.csv", index=False)
display(sw)

plt.figure(figsize=(9,5))
for col, est in [("precision","o-"),("recall","s-"),("recall_palet_roto","^--")]:
    plt.plot(sw.conf, sw[col], est, lw=2, label=col)
plt.xlabel("umbral de confianza"); plt.ylabel("valor")
plt.title("Precisión vs recall según el umbral"); plt.grid(alpha=.3); plt.legend()
plt.savefig(OUT/"barrido_conf.png", dpi=150); plt.show()

---
## 5. Exportar

Descarga todo para el repositorio y la web.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/aurion_analisis", "zip", OUT)
files.download("/content/aurion_analisis.zip")

---
## Qué mirar

**En robustez**, el codo de cada curva. Si el mAP aguanta hasta brillo 0.6 y se
desploma en 0.4, eso te da un rango operativo concreto para la web: *"el sistema
tolera variaciones de iluminación de hasta el 40%"*. Es una especificación de
despliegue.

**En errores**, abre las imágenes con prefijo `CRITICO_` y agrúpalas por causa a
mano. El recall de `palet_roto` es 0.79: uno de cada cinco se escapa, y entender
por qué es lo que convierte una métrica en un análisis.

**En el barrido**, si `recall_palet_roto` cae mucho al subir el umbral, tienes un
argumento para mantenerlo bajo y asumir más fantasmas: en control de calidad, un
falso positivo cuesta una revisión manual; un falso negativo cuesta un envío
defectuoso.